In [29]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

print("🚀 INICIANDO CREACIÓN DEL DATAFRAME MAESTRO...")
ruta_datos = '../clean/' # Asegúrate de que la ruta es correcta

# 1. CARGA DE DATOS
df_con = pd.read_csv(ruta_datos + 'conectividad_final_limpio.csv')
df_osm = pd.read_csv(ruta_datos + 'muni_station_osm_limpio.csv')
df_elec = pd.read_csv(ruta_datos + 'consumo_electrico_final_limpio.csv')
df_dem = pd.read_csv(ruta_datos + 'demografia_municipios_final.csv')
df_mig = pd.read_csv(ruta_datos + 'migracion_municipios_final_limpio.csv')
df_ren = pd.read_csv(ruta_datos + 'rentamedia_municipios_final_limpio.csv')
df_emp = pd.read_csv(ruta_datos + 'empresas_transporte_final_limpio.csv')
df_hubs = pd.read_csv(ruta_datos + 'viirsFinal_limpio.csv') 

# 2. RENOMBRES Y PREPARACIÓN DE CLAVES
df_elec = df_elec.rename(columns={'Codigo': 'LAU_ID'})
df_mig = df_mig.rename(columns={'codigo_municipio': 'LAU_ID', 'anio': 'Anio', 'cantidad (personas)': 'migracion_total'})
df_ren = df_ren.rename(columns={'codigo_municipio': 'LAU_ID', 'anio': 'Anio'})
df_dem = df_dem.rename(columns={'year': 'Anio'})

df_con['municipio_norm'] = df_con['LAU_NAME'].str.lower().str.strip()
df_dem['municipio_norm'] = df_dem['municipio'].str.lower().str.strip()

df_emp_long = df_emp.melt(id_vars=['codigo', 'nombre', 'tipo'], var_name='Anio', value_name='num_empresas_transporte')
df_emp_long = df_emp_long.rename(columns={'codigo': 'LAU_ID'})
df_emp_long['Anio'] = df_emp_long['Anio'].astype(int) 

# 🔥 LA SOLUCIÓN AL BUG DEL VIIRS 🔥
df_hubs['Anio'] = df_hubs['date'].astype(str).str[:4].astype(int)
df_hubs_anual = df_hubs.groupby(['LAU_ID', 'Anio']).agg({
    'max': 'mean', 'mean': 'mean', 'min': 'mean', 'stdDev': 'mean'
}).reset_index()

# 3. EL GRAN MERGE
df_master_ml = pd.merge(df_con, df_osm, on='LAU_ID', how='left', suffixes=('', '_osm'))
df_master_ml = pd.merge(df_master_ml, df_elec, on='LAU_ID', how='left', suffixes=('', '_elec'))
df_master_ml = pd.merge(df_master_ml, df_mig[['LAU_ID', 'Anio', 'migracion_total']], on=['LAU_ID', 'Anio'], how='left')
df_master_ml = pd.merge(df_master_ml, df_ren[['LAU_ID', 'Anio', 'pib']], on=['LAU_ID', 'Anio'], how='left')
df_master_ml = pd.merge(df_master_ml, df_emp_long[['LAU_ID', 'Anio', 'num_empresas_transporte']], on=['LAU_ID', 'Anio'], how='left')
df_master_ml = pd.merge(df_master_ml, df_hubs_anual, on=['LAU_ID', 'Anio'], how='left')
df_master_ml = pd.merge(df_master_ml, df_dem, on=['municipio_norm', 'Anio'], how='left')
df_master_ml = df_master_ml.drop(columns=['municipio_norm', 'municipio', 'LAU_NAME_osm', 'index'], errors='ignore')

print(f"✅ Dataframe Maestro. Dimensiones Reales: {df_master_ml.shape}")

🚀 INICIANDO CREACIÓN DEL DATAFRAME MAESTRO...
✅ Dataframe Maestro. Dimensiones Reales: (130096, 80)


In [30]:
# ==========================================
# 4. FEATURE ENGINEERING HÍBRIDO (Integrando en el DF parte del feature Engineering del modelo de Clasificación)
# ==========================================
print("🧬 Inyectando Feature Engineering Avanzado...")

df_master_ml['pct_viviendas_vacias'] = np.where(df_master_ml['Viviendas totales'] > 0, df_master_ml['Viviendas vacías'] / df_master_ml['Viviendas totales'], 0)
df_master_ml['densidad_poblacion'] = np.where(df_master_ml['AREA_KM2'] > 0, df_master_ml['Total'] / df_master_ml['AREA_KM2'], 0)

# El Ratio de Masculinidad (La salud demográfica)
if 'Hombres' in df_master_ml.columns and 'Mujeres' in df_master_ml.columns:
    df_master_ml['ratio_masculinidad'] = df_master_ml['Hombres'] / (df_master_ml['Mujeres'] + 0.1)
else:
    print("⚠️ Aviso: Faltan las columnas 'Hombres' y 'Mujeres' en demografía.")

# La Tasa Migratoria Relativa (El magnetismo del pueblo)
df_master_ml['tasa_migratoria_pct'] = (df_master_ml['migracion_total'] / (df_master_ml['Total'] + 1)) * 100

# La Tasa de Empresas (El impacto logístico real)
df_master_ml['tasa_empresas_pct'] = (df_master_ml['num_empresas_transporte'] / (df_master_ml['Total'] + 1)) * 1000

# Volatilidad Turística (La desviación estándar de la luz)
df_master_ml['luz_volatilidad_std'] = df_master_ml['stdDev'].fillna(0)

# Penalización por Aislamiento (Pueblos sin estación de tren cercana)
df_master_ml['mean_distance_km_to_station'] = df_master_ml['mean_distance_km_to_station'].fillna(100.0)

print(f"✅ Dataframe Maestro corregido. Dimensiones Reales: {df_master_ml.shape}")

🧬 Inyectando Feature Engineering Avanzado...
✅ Dataframe Maestro corregido. Dimensiones Reales: (130096, 86)


In [31]:
# ==========================================
# 5. PREPARACIÓN DEL SALTO TEMPORAL (PREDICCIÓN A 5 AÑOS)
# ==========================================
print("🧹 PREPARANDO EL SALTO TEMPORAL (PREDICCIÓN A 5 AÑOS)...")
df_ml = df_master_ml.copy()

# Rellenado de NaNs (Añadimos las nuevas variables relativas al relleno)
cols_fill = ['pib', 'migracion_total', 'num_empresas_transporte', 'mean', 'tasa_migratoria_pct', 'tasa_empresas_pct', 'ratio_masculinidad', 'luz_volatilidad_std']
for col in cols_fill:
    if col in df_ml.columns:
        df_ml[col] = df_ml.groupby('LAU_ID')[col].transform(lambda x: x.ffill().bfill()).fillna(0)

# 🔥 NUEVAS FEATURES: Sustituimos las absolutas por las tasas calculadas
features = [
    'Total', 'Indice_Conectividad', 'pib', 
    'tasa_migratoria_pct', 'tasa_empresas_pct', 
    'ratio_masculinidad', 'luz_volatilidad_std', 'mean', 'mean_distance_km_to_station'
]

# 🔥 EL SALTO A 5 AÑOS 🔥
df_target = df_ml[['LAU_ID', 'Anio', 'Total', 'mean']].copy()
df_target = df_target.rename(columns={'Total': 'Poblacion_5y_Futuro', 'mean': 'Luz_5y_Futuro'})
df_target['Anio_Base'] = df_target['Anio'] - 5 

# Borramos el 'Anio' del target para que Pandas no duplique nombres
df_target = df_target.drop(columns=['Anio'])

# Cruzamos la realidad actual (Anio) con la realidad futura (Anio_Base)
df_model = pd.merge(df_ml, df_target, left_on=['LAU_ID', 'Anio'], right_on=['LAU_ID', 'Anio_Base'], how='inner')
df_model = df_model.dropna(subset=['Poblacion_5y_Futuro', 'Luz_5y_Futuro'] + features)

# =========================================================
# 🔥 FILTRO RURAL: Aislamos el Target (Excluimos urbes > 50k)
# =========================================================
print("🚜 Aplicando filtro rural (municipios <= 50.000 habs)...")
df_model_rural = df_model[df_model['Total'] <= 50000].copy()

# Dividimos pasado y futuro usando SOLO el dataframe rural
anio_corte = 2017 
X_train = df_model_rural[df_model_rural['Anio'] < anio_corte][features]
X_test = df_model_rural[df_model_rural['Anio'] >= anio_corte][features]

# Nuestros DOS targets (Población y Luz) también desde el dataframe rural
y_train_pop = df_model_rural[df_model_rural['Anio'] < anio_corte]['Poblacion_5y_Futuro']
y_test_pop = df_model_rural[df_model_rural['Anio'] >= anio_corte]['Poblacion_5y_Futuro']

y_train_luz = df_model_rural[df_model_rural['Anio'] < anio_corte]['Luz_5y_Futuro']
y_test_luz = df_model_rural[df_model_rural['Anio'] >= anio_corte]['Luz_5y_Futuro']

print(f"✅ Motor temporal listo. Registros Train (Rural): {len(X_train)} | Test (Rural): {len(X_test)}")

🧹 PREPARANDO EL SALTO TEMPORAL (PREDICCIÓN A 5 AÑOS)...
🚜 Aplicando filtro rural (municipios <= 50.000 habs)...
✅ Motor temporal listo. Registros Train (Rural): 54822 | Test (Rural): 31371


In [32]:
# ==========================================
# 6. OPTIMIZACIÓN DE HIPERPARÁMETROS (GridSearch)
# ==========================================
print("🤖 INICIANDO OPTIMIZACIÓN DE HIPERPARÁMETROS (GridSearch)...")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 1. MODELO PRINCIPAL: PREDICCIÓN DE POBLACIÓN A 5 AÑOS
print("\n--- Entrenando Modelo Principal (Población) ---")
rf_pop = RandomizedSearchCV(RandomForestRegressor(random_state=42), param_distributions=param_grid, 
                            n_iter=10, cv=3, scoring='r2', n_jobs=-1, random_state=42)
rf_pop.fit(X_train, y_train_pop)

best_pop_model = rf_pop.best_estimator_
pred_pop = best_pop_model.predict(X_test)

print(f"Mejores Hiperparámetros Población: {rf_pop.best_params_}")
print(f"R2 (Población 5y): {r2_score(y_test_pop, pred_pop):.4f}")
print(f"MAE (Población 5y): {mean_absolute_error(y_test_pop, pred_pop):.2f} habitantes.")

# 2. MODELO SECUNDARIO: PREDICCIÓN DE LUZ (VIIRS) A 5 AÑOS
print("\n--- Entrenando Modelo Secundario (VIIRS) ---")
rf_luz = RandomizedSearchCV(RandomForestRegressor(random_state=42), param_distributions=param_grid, 
                            n_iter=10, cv=3, scoring='r2', n_jobs=-1, random_state=42)
rf_luz.fit(X_train, y_train_luz)

best_luz_model = rf_luz.best_estimator_
pred_luz = best_luz_model.predict(X_test)

print(f"Mejores Hiperparámetros Luz: {rf_luz.best_params_}")
print(f"R2 (Luz 5y): {r2_score(y_test_luz, pred_luz):.4f}")

🤖 INICIANDO OPTIMIZACIÓN DE HIPERPARÁMETROS (GridSearch)...

--- Entrenando Modelo Principal (Población) ---
Mejores Hiperparámetros Población: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': None}
R2 (Población 5y): 0.9975
MAE (Población 5y): 108.47 habitantes.

--- Entrenando Modelo Secundario (VIIRS) ---
Mejores Hiperparámetros Luz: {'n_estimators': 50, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': None}
R2 (Luz 5y): 0.9653


In [33]:
import joblib
import os

print("💾 GUARDANDO MODELOS PARA EL DASHBOARD EN PRODUCCIÓN...")

# Creamos una carpeta para guardar los modelos si no existe
os.makedirs('modelos_exportados', exist_ok=True)

# Exportamos el Modelo de Población
ruta_pop = 'modelos_exportados/motor_poblacion_5y.pkl'
joblib.dump(best_pop_model, ruta_pop)

# Exportamos el Modelo de Luz (VIIRS)
ruta_luz = 'modelos_exportados/motor_viirs_5y.pkl'
joblib.dump(best_luz_model, ruta_luz)

print(f"✅ ¡Modelos exportados con éxito!")
print(f" 📂 Puedes encontrarlos en la carpeta 'modelos_exportados'.")
print(" 🚀 Ya le puedes decir al equipo de Front-end que pueden cargar los archivos .pkl en su código.")

💾 GUARDANDO MODELOS PARA EL DASHBOARD EN PRODUCCIÓN...
✅ ¡Modelos exportados con éxito!
 📂 Puedes encontrarlos en la carpeta 'modelos_exportados'.
 🚀 Ya le puedes decir al equipo de Front-end que pueden cargar los archivos .pkl en su código.
